# 01｜InMemoryStore 基础操作
本例无需 API Key，也不调用模型。学习 namespace、key、value，以及新增、读取、覆盖、合并、列举和删除。

每项偏好使用独立 key；用户资料的合并另用一个 profile 文档演示。请从上到下执行。

运行前请将内核工作目录设为本章 `notebooks/`；环境变量使用 `../../../.env`，数据库使用 `../data/`（如适用）。


In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
alice_ns = ("users", "alice", "preferences")
bob_ns = ("users", "bob", "preferences")
assert store.get(alice_ns, "favorite_animal") is None


## 1. 新增和读取
namespace 像文件夹，key 像文件名，value 是 JSON 对象。`get()` 返回 Item，实际内容在 `.value`。


In [ ]:
store.put(alice_ns, "favorite_animal", {"value": "猫"})
store.put(alice_ns, "response_style", {"value": "简洁"})
item = store.get(alice_ns, "favorite_animal")
assert item is not None and item.value == {"value": "猫"}
print(item.namespace, item.key, item.value)

# 同名 key 在不同用户空间下互不影响。
assert store.get(bob_ns, "favorite_animal") is None
store.put(bob_ns, "favorite_animal", {"value": "狗"})
assert store.get(alice_ns, "favorite_animal").value["value"] == "猫"
assert store.get(bob_ns, "favorite_animal").value["value"] == "狗"


## 2. 同 namespace、同 key 的 put 是覆盖
更新某一项偏好不影响另一个 key。注意：value 内部的字段不会自动合并。


In [ ]:
store.put(alice_ns, "favorite_animal", {"value": "兔子"})
assert store.get(alice_ns, "favorite_animal").value == {"value": "兔子"}
assert store.get(alice_ns, "response_style").value == {"value": "简洁"}

profile_ns = ("users", "alice", "profile")
store.put(profile_ns, "basic", {"name": "小林", "language": "中文"})
store.put(profile_ns, "basic", {"name": "小林同学"})
assert store.get(profile_ns, "basic").value == {"name": "小林同学"}
print("覆盖后的资料：", store.get(profile_ns, "basic").value)


## 3. 需要保留旧字段时，先读再合并
这是教学用单线程示例；读后写不是原子更新，并发生产环境需要事务或相应的并发控制。


In [ ]:
store.put(profile_ns, "basic", {"name": "小林", "language": "中文"})
old_item = store.get(profile_ns, "basic")
old_value = dict(old_item.value) if old_item else {}
store.put(profile_ns, "basic", {**old_value, "name": "小林同学"})
assert store.get(profile_ns, "basic").value == {"name": "小林同学", "language": "中文"}
print("合并后的资料：", store.get(profile_ns, "basic").value)


## 4. 列举和按字段过滤
`search()` 使用 namespace **前缀**。这里传入完整的用户偏好路径；不要在面向单个用户的工具中搜索所有 users。

未配置 Embedding 索引，本例仅列举、过滤，不演示语义相似度检索。返回数量默认有限制，这里显式指定 limit。


In [ ]:
items = store.search(alice_ns, limit=10)
print([(item.key, item.value) for item in items])
assert {item.key for item in items} == {"favorite_animal", "response_style"}
matched = store.search(alice_ns, filter={"value": "兔子"}, limit=10)
assert [item.key for item in matched] == ["favorite_animal"]


## 5. 删除及内存生命周期
删除后读取返回 None，重复删除不存在的 key 也是安全的。新建的 InMemoryStore 与原实例没有共享数据；退出进程后数据不会保留。


In [ ]:
store.delete(alice_ns, "favorite_animal")
assert store.get(alice_ns, "favorite_animal") is None
store.delete(alice_ns, "favorite_animal")
assert store.get(bob_ns, "favorite_animal").value == {"value": "狗"}
assert InMemoryStore().get(alice_ns, "response_style") is None
print("全部基础断言通过。")
